In [ ]:
# 1. Instalasi Pustaka (Dependencies)
%pip install -q transformers datasets evaluate accelerate scikit-learn pandas torch

In [ ]:
# 2. Persiapan Dataset Real-World (Auto-Scraping & Auto-Labeling)
import os
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import re
import pandas as pd

def prepare_real_dataset():
    """Mengambil berita real dunia nyata & memberi label 4 kelas secara otomatis."""
    keywords = [
        "pungli truk", "pemalakan sopir", "pungutan liar jalan raya",
        "bajing loncat truk", "jalan rusak angkutan barang",
        "razia ODOL truk", "kecelakaan truk kontainer", "kemacetan jalur logistik"
    ]
    
    articles = []
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    print("=== PENGAMBILAN DATASET REAL (SCRAPING GOOGLE NEWS RSS) ===")
    
    for kw in keywords:
        encoded_kw = urllib.parse.quote(kw)
        rss_url = f"https://news.google.com/rss/search?q={encoded_kw}&hl=id&gl=ID&ceid=ID:id"
        try:
            req = urllib.request.Request(rss_url, headers=headers)
            with urllib.request.urlopen(req, timeout=10) as resp:
                xml_data = resp.read()
            root = ET.fromstring(xml_data)
            items = root.findall(".//item")
            for item in items:
                title = re.sub(r'<[^>]+>', ' ', item.findtext("title", "")).strip()
                desc = re.sub(r'<[^>]+>', ' ', item.findtext("description", "")).strip()
                title = re.sub(r'\s+', ' ', title)
                desc = re.sub(r'\s+', ' ', desc)
                full_text = f"{title}. {desc}"
                source = item.find("source").text if item.find("source") is not None else "Unknown"
                articles.append({
                    "keyword": kw, "title": title, "source": source,
                    "pub_date": item.findtext("pubDate", ""), "link": item.findtext("link", ""),
                    "text": full_text
                })
        except Exception as e:
            print(f"Error fetching '{kw}': {e}")
            
    # Hapus duplikat judul
    unique_articles = []
    seen = set()
    for a in articles:
        if a["title"] not in seen:
            seen.add(a["title"])
            unique_articles.append(a)
            
    print(f"Berhasil mengumpulkan {len(unique_articles)} artikel berita real unik.")

    # Rule-Based Auto Labeling (4 Kelas)
    def classify_text(full_str):
        full_str = full_str.lower()
        if any(re.search(r'\b' + k, full_str) for k in ["peras", "jutaan", "dipecat", "senjata", "preman", "bajing loncat", "pukul", "rampok", "ancam", "teror", "sajam", "begal"]):
            return 2 # PUNGLI_BERAT
        if any(re.search(r'\b' + k, full_str) for k in ["odol", "over dimension", "overload", "over load", "muatan berlebih", "jembatan timbang", "uji kir", "tonase", "tinggi truk", "tilang", "razia"]):
            return 3 # REGULASI_ODOL
        if any(re.search(r'\b' + k, full_str) for k in ["pungli", "uang rokok", "parkir liar", "jukir", "sumbangan", "minta uang", "karcis liar", "uang jalan"]):
            return 1 # PUNGLI_RINGAN
        return 0 # AMAN_INFORMASI

    for art in unique_articles:
        art["label"] = classify_text(f"{art['keyword']} {art['title']} {art['text']}")

    df = pd.DataFrame(unique_articles)
    df.to_csv("dataset_pungli.csv", index=False, encoding="utf-8-sig")
    print(f"Dataset berhasil disimpan ke 'dataset_pungli.csv'. Total baris: {len(df)}")
    print("\nDistribusi Label:")
    print(df['label'].value_counts().sort_index())

# Cek apakah dataset_pungli.csv sudah ada dan valid (memiliki kolom label & >50 baris)
if os.path.exists("dataset_pungli.csv"):
    try:
        df_check = pd.read_csv("dataset_pungli.csv")
        if "label" in df_check.columns and len(df_check) > 50:
            print(f"Menggunakan dataset 'dataset_pungli.csv' yang sudah ada ({len(df_check)} baris).")
        else:
            prepare_real_dataset()
    except Exception:
        prepare_real_dataset()
else:
    prepare_real_dataset()

In [ ]:
# 3. Training / Fine-Tuning Model IndoBERT (4 Kelas)
import os
import shutil
import torch
import numpy as np
import evaluate
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

# A. Cek Perangkat GPU/CPU
if torch.cuda.is_available():
    print(f"Perangkat Aktif: GPU ({torch.cuda.get_device_name(0)})")
else:
    print("Perangkat Aktif: CPU")

# B. Muat Dataset & Train/Test Split
dataset = load_dataset('csv', data_files='dataset_pungli.csv')
dataset = dataset['train'].train_test_split(test_size=0.2, seed=42)

# C. Tokenizer & Data Mapping
model_checkpoint = "indobenchmark/indobert-base-p1"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

tokenized_datasets = dataset.map(tokenize_function, batched=True)

# D. Model & Label Mapping (4 Kelas Logiway)
id2label = {
    0: "AMAN_INFORMASI",
    1: "PUNGLI_RINGAN",
    2: "PUNGLI_BERAT",
    3: "REGULASI_ODOL"
}
label2id = {
    "AMAN_INFORMASI": 0,
    "PUNGLI_RINGAN": 1,
    "PUNGLI_BERAT": 2,
    "REGULASI_ODOL": 3
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

# E. Metrik Evaluasi (Accuracy & F1-Score)
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="weighted")["f1"]
    return {"accuracy": acc, "f1": f1}

# F. Hyperparameter Training (Bersihkan Output Dir Lama Secara Manual agar Kompatibel Sempurna)
if os.path.exists("./results"):
    shutil.rmtree("./results", ignore_errors=True)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16 if torch.cuda.is_available() else 4,
    per_device_eval_batch_size=16 if torch.cuda.is_available() else 4,
    num_train_epochs=3,
    weight_decay=0.01,
    save_strategy="epoch",
    logging_steps=10,
    seed=42
)

# G. Inisialisasi Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['test'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("\nMemulai proses fine-tuning IndoBERT...")
trainer.train()

# H. Simpan Model Hasil Fine-Tuning (Bersihkan & Timpa Folder Lama Jika Ada)
model_save_path = "./indobert-pungli-classifier"
if os.path.exists(model_save_path):
    shutil.rmtree(model_save_path, ignore_errors=True)

model.save_pretrained(model_save_path)
tokenizer.save_pretrained(model_save_path)
print(f"\n[v] Model IndoBERT berhasil disimpan/ditimpa di: '{os.path.abspath(model_save_path)}'")

In [ ]:
# 4. Pengujian Prediksi Model (Inference Test)
from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./indobert-pungli-classifier",
    tokenizer="./indobert-pungli-classifier"
)

# Skenario Pengujian Laporan Real Dunia Nyata
test_samples = [
    "Jalan tol Jakarta-Cikampek terpantau lancar tanpa hambatan dan jalan mulus.",
    "Banyak pemuda nongkrong meminta uang parkir liar 20 ribu kepada sopir kontainer di simpang Marunda.",
    "Waspada preman bersenjata peras jutaan rupiah dan ancam sopir ekspedisi di lintas Sumatera.",
    "Petugas gabungan menggelar razia penimbangan truk muatan berlebih ODOL di jembatan timbang."
]

print("=== HASIL PENGUJIAN INFERENCE MODEL INDOBERT ===")
for text in test_samples:
    res = classifier(text)[0]
    print(f"Teks Laporan : {text}")
    print(f"Hasil Prediksi: {res['label']} (Confidence: {res['score']:.4f})\n")

In [ ]:
# 5. Kompresi & Ekspor Model (Menimpa File Zip & Adaptif Colab/Lokal)
import shutil
import os

zip_name = "indobert-pungli-classifier"
zip_filename = f"{zip_name}.zip"

# Hapus zip lama jika sudah ada agar bisa ditimpa
if os.path.exists(zip_filename):
    os.remove(zip_filename)

# Kompres folder model
output_zip = shutil.make_archive(zip_name, 'zip', f"./{zip_name}")
full_zip_path = os.path.abspath(output_zip)

print(f"[v] File zip model berhasil dibuat/ditimpa di: {full_zip_path}")

# Deteksi Otomatis Lingkungan (Colab vs Lokal)
try:
    from google.colab import files
    print("\nLingkungan: Google Colab terdeteksi. Memulai unduhan otomatis...")
    files.download(zip_filename)
except ImportError:
    print("\nLingkungan: Lokal terdeteksi. File zip siap digunakan secara lokal.")